In [18]:
import numpy as np
import pandas as pd


In [19]:
covid = pd.read_csv("covid_toy.csv")

In [54]:
covid.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [21]:
covid["cough"].value_counts()

cough
Mild      62
Strong    38
Name: count, dtype: int64

In [22]:
covid.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(covid.drop(columns=["has_covid"]), 
                                                    covid["has_covid"],
                                                    test_size= 0.2,
                                                   random_state= 0)

X_train.head()

,age,gender,fever,cough,city
43,22,Female,99.0,Mild,Bangalore
62,56,Female,104.0,Strong,Bangalore
3,31,Female,98.0,Mild,Kolkata
71,75,Female,104.0,Strong,Delhi
45,72,Male,99.0,Mild,Bangalore


# We are going to use OrdinalEncoder -> cough
# OneHotEncoding -> Gender, City
# SimpleImputer -> Fever

#### 1. SinpleImputer

In [57]:
# SimpleImputer
## Used to handle missing values

from sklearn.impute import SimpleImputer

SI = SimpleImputer()

# SimpleImputer() will replace NaN with a value, by default the mean.

X_train_fever = SI.fit_transform(X_train[["fever"]])
X_test_fever = SI.fit_transform(X_test[["fever"]])



In [34]:
covid["city"].value_counts()

city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64

### 2. Ordinal Encoding

In [45]:
# Ordinal Encoding -> cough
from sklearn.preprocessing import OrdinalEncoder

# Passing the order for cough in object
OE = OrdinalEncoder(categories=[["Mild", "Strong"]])

# transformed the train and test - cough column
X_train_cough = OE.fit_transform(X_train[["cough"]])
X_test_cough = OE.fit_transform(X_test[["cough"]])

X_test_cough.shape

(20, 1)

In [40]:
X_train_cough.shape

(80, 1)

### 3. OneHot Encoding

In [46]:
# OneHotEncoding -> Gender, City

from sklearn.preprocessing import OneHotEncoder

# Dropping first column to remove multicolinearity
OHE = OneHotEncoder(drop= 'first', sparse_output= False)

# Transforming Gender and city column
X_train_gender_city = OHE.fit_transform(X_train[["gender", "city"]])

X_test_gender_city = OHE.fit_transform(X_test[["gender", "city"]])

In [47]:
X_test.columns

Index(['age', 'gender', 'fever', 'cough', 'city'], dtype='object')

In [49]:
# Extracting age from train set
X_train_age = X_train.drop(columns=['gender', 'fever', 'cough', 'city']).values

#Extracting age from test set
X_test_age = X_test.drop(columns=['gender', 'fever', 'cough', 'city']).values

In [50]:
X_test_age.shape

(20, 1)

In [53]:
X_train_age.shape

(80, 1)

In [61]:
# Now need to concatenate every column(train)
X_train_updated = np.concatenate((X_train_age, X_train_fever, X_train_cough, X_train_gender_city), axis=1)
X_train_updated

array([[ 22.        ,  99.        ,   0.        ,   0.        ,
          0.        ,   0.        ,   0.        ],
       [ 56.        , 104.        ,   1.        ,   0.        ,
          0.        ,   0.        ,   0.        ],
       [ 31.        ,  98.        ,   0.        ,   0.        ,
          0.        ,   1.        ,   0.        ],
       [ 75.        , 104.        ,   1.        ,   0.        ,
          1.        ,   0.        ,   0.        ],
       [ 72.        ,  99.        ,   0.        ,   1.        ,
          0.        ,   0.        ,   0.        ],
       [ 66.        ,  99.        ,   1.        ,   1.        ,
          0.        ,   0.        ,   0.        ],
       [ 14.        , 101.        ,   1.        ,   1.        ,
          0.        ,   0.        ,   0.        ],
       [ 10.        ,  98.        ,   1.        ,   0.        ,
          0.        ,   1.        ,   0.        ],
       [ 24.        ,  98.        ,   0.        ,   1.        ,
          0.    

In [60]:
print("Age rows: ", X_train_age.shape[0])
print("X_train_fever: ", X_train_fever.shape[0])
print("X_train_cough: ", X_train_cough.shape[0])
print("X_train_gender_city: ", X_train_gender_city.shape[0])

Age rows:  80
X_train_fever:  80
X_train_cough:  80
X_train_gender_city:  80


In [62]:
# Now need to concatenate every column(test)
X_test_updated = np.concatenate((X_test_age, X_test_fever, X_test_cough, X_test_gender_city), axis=1)
X_test_updated

array([[ 19.        , 100.        ,   0.        ,   0.        ,
          0.        ,   1.        ,   0.        ],
       [ 25.        , 104.        ,   0.        ,   1.        ,
          0.        ,   0.        ,   0.        ],
       [ 42.        , 101.        ,   0.        ,   1.        ,
          1.        ,   0.        ,   0.        ],
       [ 81.        , 101.        ,   0.        ,   0.        ,
          0.        ,   0.        ,   1.        ],
       [  5.        , 102.        ,   0.        ,   1.        ,
          0.        ,   1.        ,   0.        ],
       [ 27.        , 100.        ,   0.        ,   1.        ,
          0.        ,   1.        ,   0.        ],
       [ 69.        , 103.        ,   0.        ,   0.        ,
          0.        ,   1.        ,   0.        ],
       [ 34.        ,  98.        ,   1.        ,   1.        ,
          0.        ,   1.        ,   0.        ],
       [ 60.        ,  99.        ,   0.        ,   0.        ,
          0.    

# Now Using Column Tranformer


In [68]:
# Without column transformer we need to perform lots of tasks

# Now applyinh ColumnTransformer

from sklearn.compose import ColumnTransformer

transformer = ColumnTransformer(
    transformers=[
        ("tf1", SimpleImputer(), ["fever"]),
        ("tf2", OrdinalEncoder(categories=[["Mild", "Strong"]]), ["cough"]),
        ("tf3", OneHotEncoder(drop="first", sparse_output= False), ["gender", "city"])
    ],
    remainder= "passthrough"
)

In [69]:
transformer.fit_transform(X_train)

array([[ 99.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,  22.        ],
       [104.        ,   1.        ,   0.        ,   0.        ,
          0.        ,   0.        ,  56.        ],
       [ 98.        ,   0.        ,   0.        ,   0.        ,
          1.        ,   0.        ,  31.        ],
       [104.        ,   1.        ,   0.        ,   1.        ,
          0.        ,   0.        ,  75.        ],
       [ 99.        ,   0.        ,   1.        ,   0.        ,
          0.        ,   0.        ,  72.        ],
       [ 99.        ,   1.        ,   1.        ,   0.        ,
          0.        ,   0.        ,  66.        ],
       [101.        ,   1.        ,   1.        ,   0.        ,
          0.        ,   0.        ,  14.        ],
       [ 98.        ,   1.        ,   0.        ,   0.        ,
          1.        ,   0.        ,  10.        ],
       [ 98.        ,   0.        ,   1.        ,   0.        ,
          1.    